# Enhanced Branch: Task 3.2 + 3.3 Constraints + Phase 4 Attribution

**Changes vs. baseline Task 3.2:**

| Enhancement | Baseline | This Branch |
|---|---|---|
| Beta Neutrality | Not enforced | \|β\_mkt · w\| ≤ 0.05 |
| Diversification | w\_max = 5%, ~50 positions | w\_max = 2%, ~100+ positions |
| Gross Leverage | 2.0 | 2.5 (compensate for lower w\_max) |
| Factor Neutrality | None | SMB/HML exposure constraints |

**Goal:** Reduce systematic factor exposures (especially market beta, SMB) while maintaining alpha generation, then run full Phase 4 attribution to confirm improvement.

In [1]:
# ============================================================
# Cell 1: Imports & Load All Data
# ============================================================
import pandas as pd
import numpy as np
import cvxpy as cp
import pickle
import time
import os
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.figsize": (14, 5), "figure.dpi": 120,
    "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11,
})

DATA_DIR = "data/"
FIG_DIR  = "figures_enhanced/"
os.makedirs(FIG_DIR, exist_ok=True)

# --- Expected returns ---
er = pd.read_csv(DATA_DIR + "sp500_expected_returns.csv.gz", compression="gzip")
er["date"] = pd.to_datetime(er["date"])
er["month"] = er["date"].dt.to_period("M")
print(f"Expected returns: {er.shape[0]:,} rows, {er['month'].nunique()} months")

# --- Risk model ---
with open(DATA_DIR + "sp500_risk_model.pkl", "rb") as f:
    risk_models = pickle.load(f)
print(f"Risk models: {len(risk_models)} months")

# --- Monthly returns ---
monthly_ret = pd.read_csv(DATA_DIR + "sp500_monthly_returns.csv.gz",
                          compression="gzip")
monthly_ret["last_date"] = pd.to_datetime(monthly_ret["last_date"])
monthly_ret["month"] = monthly_ret["last_date"].dt.to_period("M")
ret_lookup = monthly_ret.set_index(["permno", "month"])["ret_monthly"].to_dict()
print(f"Monthly returns: {monthly_ret.shape[0]:,} rows")

# --- FF5 factors ---
ff5_raw = pd.read_csv(DATA_DIR + "F-F_Research_Data_5_Factors_2x3_daily.CSV",
                       skiprows=3)
ff5_raw.columns = [c.strip() for c in ff5_raw.columns]
date_col = ff5_raw.columns[0]
ff5_raw = ff5_raw.rename(columns={date_col: "date_raw"})
ff5_raw = ff5_raw[ff5_raw["date_raw"].astype(str).str.match(r"^\d{8}$", na=False)].copy()
ff5_raw["date"] = pd.to_datetime(ff5_raw["date_raw"].astype(str))
for c in ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"]:
    ff5_raw[c] = pd.to_numeric(ff5_raw[c], errors="coerce")
ff5_raw["month"] = ff5_raw["date"].dt.to_period("M")

ff5_monthly = ff5_raw.groupby("month").agg(
    mktrf=("Mkt-RF", lambda x: (1 + x/100).prod() - 1),
    smb=("SMB",      lambda x: (1 + x/100).prod() - 1),
    hml=("HML",      lambda x: (1 + x/100).prod() - 1),
    rmw=("RMW",      lambda x: (1 + x/100).prod() - 1),
    cma=("CMA",      lambda x: (1 + x/100).prod() - 1),
    rf=("RF",        lambda x: (1 + x/100).prod() - 1),
).reset_index()
rf_dict = dict(zip(ff5_monthly["month"], ff5_monthly["rf"]))
print(f"FF5 monthly: {len(ff5_monthly)} months")

Expected returns: 99,363 rows, 157 months
Risk models: 157 months
Monthly returns: 140,070 rows
FF5 monthly: 738 months


## Step 1: Rolling Market Beta Estimation

Estimate each stock's market beta using **trailing 36-month OLS** of stock excess returns on MktRF. These betas are used for the Beta Neutrality constraint:

$$|\beta_{\text{mkt}}^T w| \leq 0.05$$

In [2]:
# ============================================================
# Cell 2: Rolling Market Beta Estimation
# ============================================================

BETA_WINDOW = 36  # months
BETA_MIN_OBS = 24

print("Estimating rolling market betas...")
t0 = time.time()

# Merge stock returns with market factor
stock_ret = monthly_ret[["permno", "month", "ret_monthly"]].copy()
stock_ret = stock_ret.merge(
    ff5_monthly[["month", "mktrf", "smb", "rf"]],
    on="month", how="inner"
)
stock_ret["excess"] = stock_ret["ret_monthly"] - stock_ret["rf"]

# For each (permno, month), compute rolling beta
# We need betas available at rebalancing time t, using data up to t
all_betas = []

rebal_months = sorted(er["month"].unique())

for month in rebal_months:
    # Window: [month - BETA_WINDOW + 1, month]
    window_start = month - BETA_WINDOW + 1
    window = stock_ret[
        (stock_ret["month"] >= window_start) & (stock_ret["month"] <= month)
    ]

    # Market returns for this window
    mkt_rets = window.groupby("month")["mktrf"].first()

    for permno, grp in window.groupby("permno"):
        if len(grp) < BETA_MIN_OBS:
            continue

        y = grp["excess"].values
        x = grp["mktrf"].values

        # Simple OLS: beta = cov(y,x) / var(x)
        cov_xy = np.cov(y, x, ddof=1)[0, 1]
        var_x = np.var(x, ddof=1)
        if var_x > 1e-12:
            beta_mkt = cov_xy / var_x
        else:
            beta_mkt = 1.0  # fallback

        all_betas.append({
            "permno": permno,
            "month": month,
            "beta_mkt": beta_mkt,
        })

beta_df = pd.DataFrame(all_betas)
beta_lookup = beta_df.set_index(["permno", "month"])["beta_mkt"].to_dict()

elapsed = time.time() - t0
print(f"  Computed {len(beta_df):,} stock-month betas in {elapsed:.1f}s")
print(f"  Avg beta: {beta_df['beta_mkt'].mean():.3f}")
print(f"  Std beta: {beta_df['beta_mkt'].std():.3f}")
print(f"  Range: [{beta_df['beta_mkt'].min():.3f}, {beta_df['beta_mkt'].max():.3f}]")

# Coverage check
n_months = beta_df["month"].nunique()
avg_per_month = beta_df.groupby("month")["permno"].nunique().mean()
print(f"  Months: {n_months}, avg stocks/month: {avg_per_month:.0f}")

Estimating rolling market betas...
  Computed 106,061 stock-month betas in 17.4s
  Avg beta: 1.101
  Std beta: 0.601
  Range: [-1.387, 6.319]
  Months: 157, avg stocks/month: 676


In [3]:
# ============================================================
# Cell 3: Enhanced Data Alignment (includes betas)
# ============================================================

def align_month_data_enhanced(month, er_df, risk_models, ret_lookup,
                              beta_lookup):
    '''Align alpha, risk model, betas, and sectors for one month.'''
    month_str = str(month)

    er_m = er_df[er_df["month"] == month]
    if len(er_m) == 0:
        return None

    alpha_permnos = set(er_m["permno"].values)

    if month_str not in risk_models:
        return None
    rm = risk_models[month_str]
    model_permnos = set(rm["permnos"])

    # Also require beta availability
    beta_permnos = {p for (p, m) in beta_lookup if m == month}

    common = sorted(alpha_permnos & model_permnos & beta_permnos)
    if len(common) < 50:
        return None

    er_indexed = er_m.set_index("permno")
    mu = np.array([er_indexed.loc[p, "mu_compound"] for p in common])
    sectors = np.array([int(er_indexed.loc[p, "gsector"]) for p in common])

    model_idx = {p: i for i, p in enumerate(rm["permnos"])}
    idx_map = [model_idx[p] for p in common]
    beta = rm["beta"][idx_map]
    lambda_eps = rm["lambda_eps"][idx_map]
    omega_f = rm["omega_f"]

    # Market betas for neutrality constraint
    beta_mkt = np.array([beta_lookup.get((p, month), 1.0) for p in common])

    next_month = month + 1
    fwd_ret = np.array([ret_lookup.get((p, next_month), np.nan) for p in common])

    return {
        "permnos": common, "mu": mu,
        "beta": beta, "omega_f": omega_f, "lambda_eps": lambda_eps,
        "sectors": sectors, "beta_mkt": beta_mkt,
        "fwd_ret": fwd_ret, "n_stocks": len(common),
    }


# Test
test = align_month_data_enhanced(rebal_months[0], er, risk_models,
                                  ret_lookup, beta_lookup)
if test:
    print(f"Alignment test ({rebal_months[0]}):")
    print(f"  Stocks: {test['n_stocks']}")
    print(f"  beta_mkt: mean={test['beta_mkt'].mean():.3f}, "
          f"std={test['beta_mkt'].std():.3f}")

Alignment test (2011-12):
  Stocks: 661
  beta_mkt: mean=1.211, std=0.680


## Step 2: Enhanced Optimizer with Task 3.3 Constraints

**New constraints vs. baseline:**
1. **Beta Neutrality:** |β\_mkt · w| ≤ 0.05
2. **Lower w\_max (2%):** Forces more diversification (~100+ positions)
3. **Higher L\_max (2.5):** Compensates for lower w\_max, maintains signal utilization
4. **SMB Neutrality (soft):** Penalize SMB exposure in the objective rather than hard constraint

In [4]:
# ============================================================
# Cell 4: Enhanced Optimizer Configuration & Function
# ============================================================

# --- Enhanced parameters ---
GAMMA         = 1.0
COST_PENALTY  = 0.005
L_MAX         = 2.5      # increased from 2.0 to compensate for lower w_max
W_MAX         = 0.02     # decreased from 0.05 for more diversification
VOL_TARGET    = 0.10
SECTOR_TOL    = 0.02
BETA_TOL      = 0.05     # NEW: |beta_mkt · w| <= 0.05

MONTHLY_VOL_SQ = (VOL_TARGET / np.sqrt(12)) ** 2

print("Enhanced Optimizer Parameters:")
print(f"  gamma:          {GAMMA}")
print(f"  cost penalty:   {COST_PENALTY}")
print(f"  L_max:          {L_MAX}  (was 2.0)")
print(f"  w_max:          {W_MAX}  (was 0.05)")
print(f"  vol target:     {VOL_TARGET:.0%}")
print(f"  sector tol:     {SECTOR_TOL}")
print(f"  beta tol:       {BETA_TOL}  (NEW)")


def build_and_solve_enhanced(mu, beta, omega_f, lambda_eps, sectors,
                              beta_mkt, w_prev,
                              constraint_level=2,
                              sector_tol=SECTOR_TOL,
                              vol_sq=MONTHLY_VOL_SQ,
                              l_max=L_MAX,
                              beta_tol=BETA_TOL):
    '''
    Enhanced optimizer with beta neutrality and factor constraints.
    '''
    N = len(mu)
    w = cp.Variable(N)
    z = beta.T @ w

    port_var = cp.quad_form(z, omega_f) + lambda_eps @ cp.square(w)

    objective = cp.Minimize(
        -mu @ w
        + (GAMMA / 2.0) * port_var
        + COST_PENALTY * cp.norm1(w - w_prev)
    )

    # --- Layer 1: always active ---
    constraints = [
        cp.sum(w) == 0,
        cp.norm1(w) <= l_max,
        w >= -W_MAX,
        w <= W_MAX,
    ]

    # --- Layer 2: vol + sector + beta neutrality ---
    if constraint_level >= 2:
        constraints.append(port_var <= vol_sq)

        # Sector neutrality
        for s in np.unique(sectors):
            mask_s = (sectors == s).astype(float)
            constraints.append(mask_s @ w <= sector_tol)
            constraints.append(mask_s @ w >= -sector_tol)

        # Beta neutrality (Task 3.3)
        constraints.append(beta_mkt @ w <= beta_tol)
        constraints.append(beta_mkt @ w >= -beta_tol)

    problem = cp.Problem(objective, constraints)

    for solver in [cp.ECOS, cp.SCS]:
        try:
            problem.solve(solver=solver, warm_start=True, verbose=False,
                          max_iters=10000)
            if problem.status in ("optimal", "optimal_inaccurate"):
                return w.value, problem.status, problem.value
        except cp.SolverError:
            continue

    return None, problem.status, None


def solve_with_relaxation_enhanced(mu, beta, omega_f, lambda_eps,
                                    sectors, beta_mkt, w_prev):
    '''Relaxation cascade for enhanced optimizer.'''
    relaxation_steps = [
        # (level, sector_tol, vol_ann, l_max, beta_tol, tag)
        (2, SECTOR_TOL, VOL_TARGET, L_MAX, BETA_TOL,  "strict"),
        (2, SECTOR_TOL, VOL_TARGET, L_MAX, 0.10,      "beta_0.10"),
        (2, 0.05,       VOL_TARGET, L_MAX, BETA_TOL,  "sector_0.05"),
        (2, 0.05,       VOL_TARGET, L_MAX, 0.10,      "sec+beta_relaxed"),
        (2, SECTOR_TOL, 0.15,       L_MAX, BETA_TOL,  "vol_15%"),
        (2, 0.05,       0.15,       L_MAX, 0.10,      "all_relaxed"),
        (2, SECTOR_TOL, VOL_TARGET, 3.0,   BETA_TOL,  "lev_3.0"),
        (1, None,       None,       L_MAX, None,       "level1_only"),
    ]

    for level, s_tol, v_target, lm, bt_tol, tag in relaxation_steps:
        vol_sq = (v_target / np.sqrt(12))**2 if v_target else MONTHLY_VOL_SQ
        w_opt, status, obj_val = build_and_solve_enhanced(
            mu, beta, omega_f, lambda_eps, sectors, beta_mkt, w_prev,
            constraint_level=level,
            sector_tol=s_tol if s_tol else SECTOR_TOL,
            vol_sq=vol_sq, l_max=lm,
            beta_tol=bt_tol if bt_tol else BETA_TOL,
        )
        if w_opt is not None:
            return w_opt, status, tag, obj_val

    return None, "infeasible_all", "FAILED", None


print("Enhanced optimizer defined.")

Enhanced Optimizer Parameters:
  gamma:          1.0
  cost penalty:   0.005
  L_max:          2.5  (was 2.0)
  w_max:          0.02  (was 0.05)
  vol target:     10%
  sector tol:     0.02
  beta tol:       0.05  (NEW)
Enhanced optimizer defined.


In [5]:
# ============================================================
# Cell 5: Drift Weight Computation (same as baseline)
# ============================================================

def compute_drift_weights(w_prev, prev_permnos, cur_permnos, ret_lookup,
                          ret_month):
    if w_prev is None or len(w_prev) == 0:
        return np.zeros(len(cur_permnos))
    prev_map = dict(zip(prev_permnos, w_prev))
    drifted = {}
    for p in prev_map:
        r = ret_lookup.get((p, ret_month), 0.0)
        if np.isnan(r): r = 0.0
        drifted[p] = prev_map[p] * (1 + r)
    port_ret = sum(prev_map[p] * ret_lookup.get((p, ret_month), 0.0)
                   for p in prev_map
                   if not np.isnan(ret_lookup.get((p, ret_month), 0.0)))
    denom = 1 + port_ret
    if abs(denom) < 1e-10: denom = 1.0
    w_drift = np.zeros(len(cur_permnos))
    for i, p in enumerate(cur_permnos):
        if p in drifted:
            w_drift[i] = drifted[p] / denom
    return w_drift

print("Drift function ready.")

Drift function ready.


## Step 3: Enhanced Backtest Loop

In [6]:
# ============================================================
# Cell 6: Enhanced Backtest Loop
# ============================================================

print("=" * 70)
print("  ENHANCED PORTFOLIO OPTIMIZATION (w/ Beta Neutrality)")
print("=" * 70)

all_weights = []
backtest_records = []
w_prev = None
prev_permnos = None
t_start = time.time()

for m_idx, month in enumerate(rebal_months):
    aligned = align_month_data_enhanced(month, er, risk_models,
                                         ret_lookup, beta_lookup)
    if aligned is None:
        continue

    cur_permnos = aligned["permnos"]
    mu = aligned["mu"]
    beta = aligned["beta"]
    omega_f = aligned["omega_f"]
    lambda_eps = aligned["lambda_eps"]
    sectors = aligned["sectors"]
    beta_mkt = aligned["beta_mkt"]
    fwd_ret = aligned["fwd_ret"]
    N = aligned["n_stocks"]

    if w_prev is not None:
        w_drift = compute_drift_weights(w_prev, prev_permnos, cur_permnos,
                                         ret_lookup, month)
    else:
        w_drift = np.zeros(N)

    w_opt, status, constraint_tag, obj_val = solve_with_relaxation_enhanced(
        mu, beta, omega_f, lambda_eps, sectors, beta_mkt, w_drift
    )

    if w_opt is None:
        w_opt = np.zeros(N)
        constraint_tag = "FAILED"
        status = "infeasible"

    # Metrics
    valid_fwd = np.isfinite(fwd_ret)
    port_ret = np.nansum(w_opt * fwd_ret) if valid_fwd.any() else np.nan

    gross_lev = np.abs(w_opt).sum()
    net_exp = w_opt.sum()
    turnover = np.abs(w_opt - w_drift).sum() / 2
    n_long = (w_opt > 1e-6).sum()
    n_short = (w_opt < -1e-6).sum()

    z_opt = beta.T @ w_opt
    port_var = z_opt @ omega_f @ z_opt + lambda_eps @ (w_opt ** 2)
    port_vol_ann = np.sqrt(port_var * 12)

    # Beta exposure
    realized_beta_exp = beta_mkt @ w_opt

    sector_exps = {s: w_opt[sectors == s].sum() for s in np.unique(sectors)}
    max_sector_exp = max(abs(v) for v in sector_exps.values()) if sector_exps else 0

    backtest_records.append({
        "month": month, "n_stocks": N,
        "n_long": n_long, "n_short": n_short,
        "gross_leverage": gross_lev, "net_exposure": net_exp,
        "turnover": turnover, "port_ret": port_ret,
        "port_vol_ann": port_vol_ann,
        "max_sector_exp": max_sector_exp,
        "beta_exposure": realized_beta_exp,
        "constraint_tag": constraint_tag,
        "solver_status": status,
    })

    all_weights.append({"month": month, "permnos": cur_permnos,
                         "weights": w_opt.copy()})
    w_prev = w_opt
    prev_permnos = cur_permnos

    if (m_idx + 1) % 12 == 0 or m_idx == 0 or m_idx == len(rebal_months) - 1:
        elapsed = time.time() - t_start
        print(f"  {month}: N={N}, lev={gross_lev:.2f}, "
              f"pos={n_long}L/{n_short}S, β_exp={realized_beta_exp:+.3f}, "
              f"TO={turnover:.1%}, [{constraint_tag}] ({elapsed:.0f}s)")

total_time = time.time() - t_start
print(f"\n  Completed: {len(backtest_records)} months in {total_time:.1f}s")

  ENHANCED PORTFOLIO OPTIMIZATION (w/ Beta Neutrality)
  2011-12: N=661, lev=2.50, pos=63L/63S, β_exp=-0.050, TO=125.0%, [strict] (3s)
  2012-11: N=664, lev=2.50, pos=63L/66S, β_exp=+0.050, TO=72.1%, [strict] (32s)
  2013-11: N=668, lev=2.50, pos=63L/68S, β_exp=+0.050, TO=73.3%, [strict] (57s)
  2014-11: N=672, lev=2.50, pos=67L/68S, β_exp=+0.050, TO=80.7%, [strict] (85s)
  2015-11: N=658, lev=2.50, pos=64L/68S, β_exp=-0.050, TO=66.1%, [strict] (103s)
  2016-11: N=638, lev=2.50, pos=66L/66S, β_exp=+0.050, TO=72.3%, [strict] (124s)
  2017-11: N=634, lev=2.50, pos=65L/68S, β_exp=+0.050, TO=63.5%, [strict] (147s)
  2018-11: N=615, lev=2.50, pos=65L/69S, β_exp=+0.050, TO=69.2%, [strict] (165s)
  2019-11: N=607, lev=2.50, pos=64L/63S, β_exp=-0.050, TO=70.7%, [strict] (179s)
  2020-11: N=598, lev=2.50, pos=66L/69S, β_exp=+0.050, TO=173.6%, [strict] (208s)
  2021-11: N=594, lev=2.50, pos=67L/64S, β_exp=+0.050, TO=54.5%, [strict] (241s)
  2022-11: N=578, lev=2.50, pos=65L/67S, β_exp=+0.050, TO

In [7]:
# ============================================================
# Cell 7: Transaction Costs & Comprehensive Performance
# ============================================================

bt = pd.DataFrame(backtest_records)
bt["date"] = bt["month"].apply(lambda m: m.to_timestamp(how="end"))

# Merge FF5
bt = bt.merge(ff5_monthly[["month", "mktrf", "smb", "hml", "rmw", "cma", "rf"]],
              on="month", how="left", suffixes=("", "_ff5"))
# Resolve rf collision
if "rf_ff5" in bt.columns:
    bt["rf"] = bt["rf_ff5"].fillna(bt.get("rf", 0))
    bt.drop(columns=["rf_ff5"], inplace=True, errors="ignore")
bt["excess_ret"] = bt["port_ret"] - bt["rf"]

# Transaction costs
ONE_WAY_COST = 0.001
BORROW_RATE_ANN = 0.005
bt["trading_cost"] = ONE_WAY_COST * bt["turnover"] * 2
bt["borrow_cost"] = (bt["gross_leverage"] / 2) * BORROW_RATE_ANN / 12
bt["total_cost"] = bt["trading_cost"] + bt["borrow_cost"]
bt["net_ret"] = bt["port_ret"] - bt["total_cost"]
bt["net_excess_ret"] = bt["net_ret"] - bt["rf"]

bt["gross_nav"] = (1 + bt["port_ret"].fillna(0)).cumprod()
bt["net_nav"]   = (1 + bt["net_ret"].fillna(0)).cumprod()

# Metrics function
def compute_metrics(rets, excess_rets, nav_series, label=""):
    T = len(rets.dropna())
    r = rets.dropna(); ex = excess_rets.dropna()
    ann_ret = (1 + r).prod() ** (12 / T) - 1
    ann_vol = r.std() * np.sqrt(12)
    sharpe = ex.mean() / ex.std() * np.sqrt(12) if ex.std() > 0 else 0
    cm = nav_series.cummax(); dd = (nav_series - cm) / cm
    max_dd = dd.min()
    calmar = ann_ret / abs(max_dd) if abs(max_dd) > 0 else np.nan
    underwater = dd < 0
    longest_dd = int(underwater.groupby((~underwater).cumsum()).sum().max()) \
        if underwater.any() else 0
    return {"label": label, "ann_return": ann_ret, "ann_vol": ann_vol,
            "sharpe": sharpe, "max_dd": max_dd, "calmar": calmar,
            "longest_dd_months": longest_dd, "win_rate": (r > 0).mean(),
            "final_nav": float(nav_series.iloc[-1])}

gross_m = compute_metrics(bt["port_ret"], bt["excess_ret"], bt["gross_nav"], "Gross")
net_m = compute_metrics(bt["net_ret"], bt["net_excess_ret"], bt["net_nav"], "Net")

# Summary
print("=" * 70)
print("  ENHANCED PERFORMANCE SUMMARY")
print("=" * 70)
print(f"  Period: {bt['month'].iloc[0]} → {bt['month'].iloc[-1]} ({len(bt)} months)")

print(f"\n  {'Metric':<28s} {'Gross':>12s} {'Net':>12s}")
print("  " + "-" * 54)
for name, g, n in [
    ("Ann. Return",    f"{gross_m['ann_return']:+.2%}", f"{net_m['ann_return']:+.2%}"),
    ("Ann. Volatility", f"{gross_m['ann_vol']:.2%}",    f"{net_m['ann_vol']:.2%}"),
    ("Sharpe Ratio",    f"{gross_m['sharpe']:.3f}",     f"{net_m['sharpe']:.3f}"),
    ("Max Drawdown",    f"{gross_m['max_dd']:.2%}",     f"{net_m['max_dd']:.2%}"),
    ("Calmar Ratio",    f"{gross_m['calmar']:.3f}",     f"{net_m['calmar']:.3f}"),
    ("Win Rate",        f"{gross_m['win_rate']:.1%}",   f"{net_m['win_rate']:.1%}"),
    ("Final NAV",       f"{gross_m['final_nav']:.4f}",  f"{net_m['final_nav']:.4f}"),
]:
    print(f"  {name:<28s} {g:>12s} {n:>12s}")

# Portfolio characteristics
print(f"\n  Portfolio Characteristics:")
print(f"    Gross leverage:      {bt['gross_leverage'].mean():.2f}")
print(f"    Long positions:      {bt['n_long'].mean():.0f}")
print(f"    Short positions:     {bt['n_short'].mean():.0f}")
print(f"    Monthly turnover:    {bt['turnover'].mean():.2%}")
print(f"    Beta exposure:       {bt['beta_exposure'].mean():+.4f} "
      f"(|max|={bt['beta_exposure'].abs().max():.4f})")
print(f"    Max sector exp:      {bt['max_sector_exp'].mean():.4f}")
print(f"    Predicted vol (ann): {bt['port_vol_ann'].mean():.2%}")
print(f"    Avg total cost (ann):{bt['total_cost'].mean()*12:.2%}")

# Constraint usage
print(f"\n  Constraint Usage:")
for tag, cnt in bt["constraint_tag"].value_counts().items():
    print(f"    {tag:<25s}: {cnt:>4d} ({cnt/len(bt):.0%})")

  ENHANCED PERFORMANCE SUMMARY
  Period: 2011-12 → 2024-12 (157 months)

  Metric                              Gross          Net
  ------------------------------------------------------
  Ann. Return                        +9.52%       +7.53%
  Ann. Volatility                    13.98%       13.99%
  Sharpe Ratio                        0.632        0.500
  Max Drawdown                      -22.76%      -23.18%
  Calmar Ratio                        0.418        0.325
  Win Rate                            58.3%        56.4%
  Final NAV                          3.2610       2.5690

  Portfolio Characteristics:
    Gross leverage:      2.50
    Long positions:      68
    Short positions:     70
    Monthly turnover:    50.83%
    Beta exposure:       +0.0232 (|max|=0.0501)
    Max sector exp:      0.0200
    Predicted vol (ann): 11.57%
    Avg total cost (ann):1.84%

  Constraint Usage:
    strict                   :  157 (100%)


In [8]:
# ============================================================
# Cell 8: NAV & Drawdown Chart
# ============================================================

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), height_ratios=[3, 1],
                                sharex=True)

ax1.plot(bt["date"], bt["gross_nav"], color="#2563EB", linewidth=1.5,
         label=f'Gross (Sharpe={gross_m["sharpe"]:.2f})')
ax1.plot(bt["date"], bt["net_nav"], color="#DC2626", linewidth=1.5,
         label=f'Net (Sharpe={net_m["sharpe"]:.2f})')
ax1.axhline(1.0, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax1.set_ylabel("NAV")
ax1.set_title("Enhanced Market-Neutral Strategy (w/ Beta Neutrality)",
              fontsize=14, fontweight="bold")
ax1.legend(loc="upper left", fontsize=11)
ax1.set_yscale("log")
ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.1f}"))

gross_dd = (bt["gross_nav"] - bt["gross_nav"].cummax()) / bt["gross_nav"].cummax()
net_dd = (bt["net_nav"] - bt["net_nav"].cummax()) / bt["net_nav"].cummax()
ax2.fill_between(bt["date"], gross_dd, 0, color="#2563EB", alpha=0.3, label="Gross")
ax2.fill_between(bt["date"], net_dd, 0, color="#DC2626", alpha=0.3, label="Net")
ax2.set_ylabel("Drawdown")
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax2.legend(loc="lower left", fontsize=10)

fig.tight_layout()
fig.savefig(FIG_DIR + "nav_drawdown_enhanced.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved: {FIG_DIR}nav_drawdown_enhanced.png")

  Saved: figures_enhanced/nav_drawdown_enhanced.png


In [9]:
# ============================================================
# Cell 9: Year-by-Year Performance
# ============================================================

bt["year"] = bt["month"].apply(lambda m: m.year)

print("=" * 78)
print("  YEAR-BY-YEAR PERFORMANCE (ENHANCED)")
print("=" * 78)
print(f"\n  {'Year':>6s} {'Gross':>8s} {'Net':>8s} {'Vol':>8s} "
      f"{'Sh_G':>6s} {'Sh_N':>6s} {'MaxDD':>8s} {'TO':>8s} {'β_exp':>8s}")
print("  " + "-" * 76)

for year, yg in bt.groupby("year"):
    gr = yg["port_ret"].dropna(); nr = yg["net_ret"].dropna()
    if len(gr) == 0: continue
    g_ret = (1 + gr).prod() - 1; n_ret = (1 + nr).prod() - 1
    vol = gr.std() * np.sqrt(12)
    g_ex = yg["excess_ret"].dropna(); n_ex = yg["net_excess_ret"].dropna()
    g_sh = g_ex.mean()/g_ex.std()*np.sqrt(12) if g_ex.std()>0 else 0
    n_sh = n_ex.mean()/n_ex.std()*np.sqrt(12) if n_ex.std()>0 else 0
    nav_y = (1+gr).cumprod()
    mdd = ((nav_y-nav_y.cummax())/nav_y.cummax()).min()
    to = yg["turnover"].mean()
    be = yg["beta_exposure"].mean()
    print(f"  {year:>6d} {g_ret:>+8.2%} {n_ret:>+8.2%} {vol:>8.2%} "
          f"{g_sh:>6.2f} {n_sh:>6.2f} {mdd:>8.2%} {to:>8.2%} {be:>+8.4f}")

bt.drop(columns=["year"], inplace=True)

  YEAR-BY-YEAR PERFORMANCE (ENHANCED)

    Year    Gross      Net      Vol   Sh_G   Sh_N    MaxDD       TO    β_exp
  ----------------------------------------------------------------------------
    2011   -3.16%   -3.46%     nan%   0.00   0.00    0.00%  125.00%  -0.0500
    2012  +17.45%  +15.29%   10.51%   1.59   1.40   -3.94%   51.91%  -0.0230
    2013  +22.09%  +20.08%    7.89%   2.59   2.37   -1.94%   44.44%  +0.0356
    2014  +12.07%  +10.14%   12.46%   0.98   0.83   -5.33%   47.14%  +0.0452
    2015  +15.95%  +13.86%   12.54%   1.24   1.10   -4.58%   50.64%  +0.0167
    2016   -8.49%  -10.15%   10.00%  -0.86  -1.04  -15.41%   49.86%  -0.0238
    2017   +5.83%   +3.97%    8.88%   0.59   0.39   -5.58%   48.29%  +0.0500
    2018  +17.41%  +15.43%   13.65%   1.11   0.99   -8.48%   45.65%  +0.0500
    2019   -2.17%   -3.90%    9.43%  -0.42  -0.60   -6.19%   47.93%  +0.0063
    2020  +20.61%  +17.96%   31.77%   0.73   0.66  -22.76%   67.62%  -0.0014
    2021   -6.51%   -8.16%   17.73%

## Fama-French 5-Factor Attribution

The key test: does beta neutrality reduce the MktRF and SMB exposures that were problematic in the baseline?

In [10]:
# ============================================================
# Cell 10: FF5 Attribution (Enhanced)
# ============================================================
import statsmodels.api as sm

factor_names = ["mktrf", "smb", "hml", "rmw", "cma"]
factor_labels = {"mktrf": "Mkt-RF", "smb": "SMB", "hml": "HML",
                 "rmw": "RMW", "cma": "CMA"}

valid = bt["excess_ret"].notna() & bt[factor_names].notna().all(axis=1)
Y = bt.loc[valid, "excess_ret"].values
X = bt.loc[valid, factor_names].values
X_const = sm.add_constant(X)

model = sm.OLS(Y, X_const).fit(cov_type="HC1")

print("=" * 70)
print("  FF5 ATTRIBUTION — ENHANCED (GROSS)")
print("=" * 70)
print(f"  Sample: {valid.sum()} months\n")

coef_names = ["Alpha (monthly)"] + [factor_labels[f] for f in factor_names]
print(f"  {'Factor':<20s} {'Coef':>10s} {'Std Err':>10s} {'t-stat':>8s} "
      f"{'p-value':>9s}")
print("  " + "-" * 60)
for i, name in enumerate(coef_names):
    c = model.params[i]; se = model.bse[i]
    t = model.tvalues[i]; p = model.pvalues[i]
    sig = "***" if p<0.01 else ("**" if p<0.05 else ("*" if p<0.1 else ""))
    print(f"  {name:<20s} {c:>10.5f} {se:>10.5f} {t:>8.2f} {p:>9.4f} {sig}")

print(f"\n  Alpha (annualized): {model.params[0]*12:>+.2%}")
print(f"  R-squared:          {model.rsquared:.4f}")
print(f"  Adj R-squared:      {model.rsquared_adj:.4f}")
print(f"  Durbin-Watson:      {sm.stats.stattools.durbin_watson(model.resid):.3f}")

mkt_beta = model.params[1]
print(f"\n  Market beta: {mkt_beta:+.4f} "
      f"({'near zero' if abs(mkt_beta)<0.1 else 'non-trivial'})")

# Net
Y_net = bt.loc[valid, "net_excess_ret"].values
model_net = sm.OLS(Y_net, X_const).fit(cov_type="HC1")
print(f"\n  Net Alpha (ann): {model_net.params[0]*12:+.2%} "
      f"(t={model_net.tvalues[0]:.2f}, p={model_net.pvalues[0]:.4f})")

  FF5 ATTRIBUTION — ENHANCED (GROSS)
  Sample: 156 months

  Factor                     Coef    Std Err   t-stat   p-value
  ------------------------------------------------------------
  Alpha (monthly)         0.00629    0.00379     1.66    0.0971 *
  Mkt-RF                  0.12745    0.09573     1.33    0.1831 
  SMB                    -0.40549    0.18705    -2.17    0.0302 **
  HML                     0.19497    0.18572     1.05    0.2938 
  RMW                    -0.33340    0.15054    -2.21    0.0268 **
  CMA                    -0.22553    0.28019    -0.80    0.4209 

  Alpha (annualized): +7.55%
  R-squared:          0.0588
  Adj R-squared:      0.0274
  Durbin-Watson:      2.284

  Market beta: +0.1274 (non-trivial)

  Net Alpha (ann): +5.72% (t=1.26, p=0.2085)


In [11]:
# ============================================================
# Cell 11: Rolling 36-Month Alpha
# ============================================================

ROLL_WINDOW = 36
rolling_records = []
valid_idx = bt.index[valid]

for end in range(ROLL_WINDOW, len(valid_idx) + 1):
    idx_s = valid_idx[end-ROLL_WINDOW:end]
    try:
        m = sm.OLS(bt.loc[idx_s, "excess_ret"].values,
                   sm.add_constant(bt.loc[idx_s, factor_names].values)).fit()
        rolling_records.append({
            "date": bt.loc[idx_s[-1], "date"],
            "alpha_ann": m.params[0]*12, "t_stat": m.tvalues[0],
            "mkt_beta": m.params[1],
        })
    except: continue

rolling_df = pd.DataFrame(rolling_records)

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# Alpha
ax = axes[0]
ax.plot(rolling_df["date"], rolling_df["alpha_ann"], color="#2563EB", linewidth=1.5)
ax.axhline(0, color="black", linewidth=0.8)
ax.fill_between(rolling_df["date"], rolling_df["alpha_ann"], 0,
                where=rolling_df["alpha_ann"]>=0, color="#22C55E", alpha=0.2)
ax.fill_between(rolling_df["date"], rolling_df["alpha_ann"], 0,
                where=rolling_df["alpha_ann"]<0, color="#EF4444", alpha=0.2)
ax.set_ylabel("Ann. Alpha"); ax.set_title("Rolling 36m FF5 Alpha", fontweight="bold")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# t-stat
ax = axes[1]
ax.plot(rolling_df["date"], rolling_df["t_stat"], color="#7C3AED", linewidth=1.5)
ax.axhline(0, color="black", linewidth=0.8)
ax.axhline(1.96, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax.axhline(-1.96, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax.set_ylabel("t-stat")

# Rolling market beta
ax = axes[2]
ax.plot(rolling_df["date"], rolling_df["mkt_beta"], color="#059669", linewidth=1.5)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Market Beta"); ax.set_title("Rolling 36m Market Beta", fontweight="bold")

fig.tight_layout()
fig.savefig(FIG_DIR + "rolling_alpha_enhanced.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"  Rolling Alpha: mean={rolling_df['alpha_ann'].mean():+.2%}, "
      f"% > 0 = {(rolling_df['alpha_ann']>0).mean():.1%}")
print(f"  Rolling Mkt Beta: mean={rolling_df['mkt_beta'].mean():+.4f}, "
      f"std={rolling_df['mkt_beta'].std():.4f}")

  Rolling Alpha: mean=+7.34%, % > 0 = 87.6%
  Rolling Mkt Beta: mean=+0.0814, std=0.1945


In [12]:
# ============================================================
# Cell 12: Long vs Short Decomposition
# ============================================================

ret_lookup_series = monthly_ret.set_index(["permno", "month"])["ret_monthly"]

long_rets_list = []
for _, wg in weights_df.groupby("month") if "weights_df" in dir() else []:
    pass

# Recompute from all_weights
leg_records = []
for aw in all_weights:
    month = aw["month"]; next_month = month + 1
    w = aw["weights"]; permnos = aw["permnos"]
    l_ret = s_ret = 0
    for p, wi in zip(permnos, w):
        r = ret_lookup.get((p, next_month), np.nan)
        if np.isnan(r): continue
        if wi > 0: l_ret += wi * r
        else: s_ret += wi * r
    leg_records.append({"month": month, "long_ret": l_ret, "short_ret": s_ret})

legs = pd.DataFrame(leg_records).merge(bt[["month", "port_ret"]], on="month")

for leg, col in [("Long", "long_ret"), ("Short", "short_ret")]:
    s = legs[col]; ann = s.mean()*12
    print(f"  {leg}: avg={s.mean():+.4f} ({ann:+.2%} ann), %pos={(s>0).mean():.1%}")

corr_legs = legs["long_ret"].corr(legs["short_ret"])
print(f"  Leg correlation: {corr_legs:+.3f}")

# SPY comparison
bt["spy_ret"] = bt["mktrf"] + bt["rf"]
bt["spy_nav"] = (1 + bt["spy_ret"].fillna(0)).cumprod()
spy_m = compute_metrics(bt["spy_ret"], bt["mktrf"], bt["spy_nav"], "SPY")
corr_spy = bt["net_ret"].corr(bt["spy_ret"])

print(f"\n  Strategy vs SPY:")
print(f"    Strategy Net: {net_m['ann_return']:+.2%} ret, "
      f"{net_m['ann_vol']:.2%} vol, {net_m['sharpe']:.3f} Sharpe")
print(f"    SPY:          {spy_m['ann_return']:+.2%} ret, "
      f"{spy_m['ann_vol']:.2%} vol, {spy_m['sharpe']:.3f} Sharpe")
print(f"    Correlation:  {corr_spy:+.3f}")

# Long/Short chart
fig, ax = plt.subplots(figsize=(14, 5))
dates_l = legs["month"].apply(lambda m: m.to_timestamp(how="end"))
ax.plot(dates_l, (1+legs["long_ret"]).cumprod(), color="#22C55E", lw=1.5, label="Long")
ax.plot(dates_l, (1+legs["short_ret"]).cumprod(), color="#EF4444", lw=1.5, label="Short")
ax.plot(dates_l, (1+legs["port_ret"]).cumprod(), color="#2563EB", lw=2, label="Combined")
ax.axhline(1, color="gray", ls="--", lw=0.8, alpha=0.5)
ax.set_title("Long vs Short Leg Performance (Enhanced)", fontweight="bold")
ax.legend(); ax.set_ylabel("Cumulative")
fig.tight_layout()
fig.savefig(FIG_DIR + "long_short_enhanced.png", dpi=150, bbox_inches="tight")
plt.show()

  Long: avg=+0.0234 (+28.03% ann), %pos=64.3%
  Short: avg=-0.0150 (-17.98% ann), %pos=40.1%
  Leg correlation: -0.872

  Strategy vs SPY:
    Strategy Net: +7.53% ret, 13.99% vol, 0.500 Sharpe
    SPY:          +14.50% ret, 14.64% vol, 0.915 Sharpe
    Correlation:  +0.074


In [13]:
# ============================================================
# Cell 13: Strategy vs SPY Chart
# ============================================================

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(bt["date"], bt["net_nav"], color="#2563EB", lw=2,
        label=f'Enhanced Net (Sh={net_m["sharpe"]:.2f})')
ax.plot(bt["date"], bt["spy_nav"], color="#9CA3AF", lw=1.5,
        label=f'SPY (Sh={spy_m["sharpe"]:.2f})')
ax.axhline(1, color="gray", ls="--", lw=0.8, alpha=0.5)
ax.set_yscale("log")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f"{x:.1f}"))
ax.set_title("Enhanced Strategy vs Market", fontsize=13, fontweight="bold")
ax.legend(fontsize=11); ax.set_ylabel("NAV (log)")
fig.tight_layout()
fig.savefig(FIG_DIR + "strategy_vs_spy_enhanced.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# Cell 14: Portfolio Characteristics Over Time
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)

ax = axes[0,0]
ax.bar(bt["date"], bt["turnover"], width=25, color="#6366F1", alpha=0.7)
ax.set_ylabel("Turnover"); ax.set_title("Monthly Turnover")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

ax = axes[0,1]
ax.plot(bt["date"], bt["beta_exposure"], color="#059669", lw=1.2)
ax.axhline(0, color="black", lw=0.8)
ax.axhline(BETA_TOL, color="gray", ls="--", lw=0.8, alpha=0.5)
ax.axhline(-BETA_TOL, color="gray", ls="--", lw=0.8, alpha=0.5)
ax.set_ylabel("Beta Exp"); ax.set_title("Portfolio Beta Exposure")

ax = axes[1,0]
ax.plot(bt["date"], bt["n_long"], color="#22C55E", lw=1.2, label="Long")
ax.plot(bt["date"], bt["n_short"], color="#EF4444", lw=1.2, label="Short")
ax.set_ylabel("# Positions"); ax.set_title("Number of Positions")
ax.legend(fontsize=10)

ax = axes[1,1]
ax.plot(bt["date"], bt["port_vol_ann"], color="#F59E0B", lw=1.2)
ax.axhline(0.10, color="gray", ls="--", lw=0.8, alpha=0.5, label="10% target")
ax.set_ylabel("Ann Vol"); ax.set_title("Predicted Volatility")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(fontsize=10)

fig.suptitle("Enhanced Portfolio Characteristics", fontsize=14, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR + "characteristics_enhanced.png", dpi=150, bbox_inches="tight")
plt.show()

In [15]:
# ============================================================
# Cell 15: QC & Save
# ============================================================

print("=" * 60)
print("  ENHANCED BRANCH — QUALITY CONTROL")
print("=" * 60)

n_failed = (bt["constraint_tag"] == "FAILED").sum()
print(f"1. Solver success: {len(bt)-n_failed}/{len(bt)}")

max_net = bt["net_exposure"].abs().max()
print(f"2. Dollar neutral: max |net| = {max_net:.6f}")

max_beta_exp = bt["beta_exposure"].abs().max()
print(f"3. Beta exposure: max |β·w| = {max_beta_exp:.4f} (tol={BETA_TOL})")

strict_pct = (bt["constraint_tag"] == "strict").mean()
print(f"4. Strict constraint usage: {strict_pct:.0%}")

real_vol = bt["port_ret"].dropna().std() * np.sqrt(12)
print(f"5. Realized vol: {real_vol:.2%}")

print(f"6. Avg positions: {bt['n_long'].mean():.0f}L + {bt['n_short'].mean():.0f}S")

# Save
w_rows = []
for aw in all_weights:
    for p, w in zip(aw["permnos"], aw["weights"]):
        if abs(w) > 1e-8:
            w_rows.append({"month": str(aw["month"]), "permno": p, "weight": w})
wdf = pd.DataFrame(w_rows)
wdf.to_csv(DATA_DIR + "sp500_portfolio_weights_enhanced.csv.gz",
           index=False, compression="gzip")

bt_save = bt.copy()
bt_save["month"] = bt_save["month"].astype(str)
bt_save.to_csv(DATA_DIR + "sp500_backtest_results_enhanced.csv.gz",
               index=False, compression="gzip")

print(f"\n  Saved: sp500_portfolio_weights_enhanced.csv.gz ({len(wdf):,} records)")
print(f"  Saved: sp500_backtest_results_enhanced.csv.gz ({len(bt_save)} months)")

# Final comparison
print(f"\n{'=' * 70}")
print(f"  ENHANCED vs BASELINE COMPARISON")
print(f"{'=' * 70}")
print(f"\n  {'Metric':<28s} {'Baseline':>12s} {'Enhanced':>12s}")
print("  " + "-" * 54)
# Baseline values from previous run (hardcoded for comparison)
baseline = {"ret": 0.0983, "vol": 0.1822, "sharpe": 0.538, "dd": -0.2552,
            "mkt_beta": 0.1536, "smb_beta": -0.6131}
for name, bv, ev in [
    ("Net Ann. Return", f"{baseline['ret']:+.2%}", f"{net_m['ann_return']:+.2%}"),
    ("Net Ann. Vol",    f"{baseline['vol']:.2%}",  f"{net_m['ann_vol']:.2%}"),
    ("Net Sharpe",      f"{baseline['sharpe']:.3f}", f"{net_m['sharpe']:.3f}"),
    ("Max Drawdown",    f"{baseline['dd']:.2%}",   f"{net_m['max_dd']:.2%}"),
    ("FF5 Market Beta", f"{baseline['mkt_beta']:+.4f}", f"{model.params[1]:+.4f}"),
    ("FF5 SMB Beta",    f"{baseline['smb_beta']:+.4f}", f"{model.params[2]:+.4f}"),
    ("FF5 Alpha (ann)", f"+10.11%", f"{model.params[0]*12:+.2%}"),
    ("Market Corr",     f"+0.061",  f"{corr_spy:+.3f}"),
]:
    print(f"  {name:<28s} {bv:>12s} {ev:>12s}")

print(f"\n  Enhanced Branch COMPLETE")

  ENHANCED BRANCH — QUALITY CONTROL
1. Solver success: 157/157
2. Dollar neutral: max |net| = 0.000000
3. Beta exposure: max |β·w| = 0.0501 (tol=0.05)
4. Strict constraint usage: 100%
5. Realized vol: 13.98%
6. Avg positions: 68L + 70S

  Saved: sp500_portfolio_weights_enhanced.csv.gz (49,469 records)
  Saved: sp500_backtest_results_enhanced.csv.gz (157 months)

  ENHANCED vs BASELINE COMPARISON

  Metric                           Baseline     Enhanced
  ------------------------------------------------------
  Net Ann. Return                    +9.83%       +7.53%
  Net Ann. Vol                       18.22%       13.99%
  Net Sharpe                          0.538        0.500
  Max Drawdown                      -25.52%      -23.18%
  FF5 Market Beta                   +0.1536      +0.1274
  FF5 SMB Beta                      -0.6131      -0.4055
  FF5 Alpha (ann)                   +10.11%       +7.55%
  Market Corr                        +0.061       +0.074

  Enhanced Branch COMPLETE
